In [315]:
%matplotlib
import torch
import numpy as np
import matplotlib.pyplot as plt

Using matplotlib backend: TkAgg


In [316]:
com_0 = torch.tensor([[0, 0, 0.3], [0, 0, 0.3]])

In [422]:
com_f = torch.tensor([[0.02823, 0.07039, 0.1932], [-0.08823, -0.03039, 0.3]])
comd_f = torch.tensor([[0.67559, 1.6845, 1.55862], [0.57559, 1.545, 1.35862]])
t_th = torch.tensor([[0.65], [0.7]])

In [423]:
w = torch.stack((
    com_0.unsqueeze(1),
    com_0.unsqueeze(1),
    (com_f - (1. / 3.) * t_th * comd_f).unsqueeze(1),
    com_f.unsqueeze(1)), dim=2).squeeze()

In [424]:
w

tensor([[[ 0.0000,  0.0000,  0.3000],
         [ 0.0000,  0.0000,  0.3000],
         [-0.1181, -0.2946, -0.1445],
         [ 0.0282,  0.0704,  0.1932]],

        [[ 0.0000,  0.0000,  0.3000],
         [ 0.0000,  0.0000,  0.3000],
         [-0.2225, -0.3909, -0.0170],
         [-0.0882, -0.0304,  0.3000]]])

In [425]:
t_ex = 0.7

In [426]:
t = t_ex / t_th

In [427]:
t

tensor([[1.0769],
        [1.0000]])

In [428]:
bezier_curve = torch.cat((
    (1.0) * (t**0) * (1 - t)**(3 - 0),
    (3.0) * (t**1) * (1 - t)**(3 - 1),
    (3.0) * (t**2) * (1 - t)**(3 - 2),
    (1.0) * (t**3) * (1 - t)**(3 - 3),
), dim=-1)

In [429]:
bezier_curve

tensor([[-4.5517e-04,  1.9117e-02, -2.6764e-01,  1.2490e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00]])

In [430]:
w_x = w[..., 0]
w_y = w[..., 1]
w_z = w[..., 2]

In [431]:
com = torch.cat((
    (w_x * bezier_curve).sum(dim=-1).reshape(-1, 1),
    (w_y * bezier_curve).sum(dim=-1).reshape(-1, 1),
    (w_z * bezier_curve).sum(dim=-1).reshape(-1, 1)), dim=1)
com

tensor([[ 0.0669,  0.1668,  0.2856],
        [-0.0882, -0.0304,  0.3000]])

In [432]:
com_f - com

tensor([[-0.0386, -0.0964, -0.0924],
        [ 0.0000,  0.0000,  0.0000]])

In [433]:
t_ex > t_th

tensor([[ True],
        [False]])

In [434]:
def BezierTrajectory(com_0: torch.Tensor, com_f: torch.Tensor, comd_f: torch.Tensor, t_th: torch.Tensor, t_ex: float):

    w = torch.stack((
        com_0.unsqueeze(1),
        com_0.unsqueeze(1),
        (com_f - (1. / 3.) * t_th * comd_f).unsqueeze(1),
        com_f.unsqueeze(1)), dim=2).squeeze()

    t = t_ex / t_th

    bezier_curve = torch.cat((
        (1.0) * (t**0) * (1 - t)**(3 - 0),
        (3.0) * (t**1) * (1 - t)**(3 - 1),
        (3.0) * (t**2) * (1 - t)**(3 - 2),
        (1.0) * (t**3) * (1 - t)**(3 - 3),
    ), dim=-1)

    bezier_point = torch.cat((
        (w[..., 0] * bezier_curve).sum(dim=-1).reshape(-1, 1),
        (w[..., 1] * bezier_curve).sum(dim=-1).reshape(-1, 1),
        (w[..., 2] * bezier_curve).sum(dim=-1).reshape(-1, 1)), dim=1)

    return bezier_point

In [435]:
BezierTrajectory(com_0, com_f, comd_f, t_th, 0.65)

tensor([[ 0.0282,  0.0704,  0.1932],
        [-0.1118, -0.0966,  0.2414]])

In [436]:
traj = torch.stack([BezierTrajectory(com_0, com_f, comd_f, t_th, t) for t in torch.arange(0,1,0.005)], dim=1)

In [437]:
traj.shape

torch.Size([2, 200, 3])

In [438]:
# plt.plot(traj[0][:,0],traj[0][:,2])
# plt.scatter(com_f[0, 0],com_f[0, 2], color='red')

In [440]:
fig = plt.figure(figsize=(10, 10))
ax = plt.axes(projection='3d')
ax.set_zlim(0, 0.35)
ax.set_xlim(-0.35, 0.35)
ax.set_ylim(-0.35, 0.35)
ax.scatter(traj[..., 0], traj[..., 1], traj[..., 2], color='blue')
ax.scatter(com_0[:, 0], com_0[:, 1], com_0[:, 2], alpha=1, linewidths=5)
ax.scatter(com_f[:, 0], com_f[:, 1], com_f[:, 2], alpha=1, linewidths=5)
plt.show()